# 05 — Export Fine-Tuned Model

Merges LoRA adapters into full model, quantizes for deployment,
and registers in MLflow for production use.


In [ ]:
# Cell 1: Merge LoRA → full model
from unsloth import FastVisionModel

model, tokenizer = FastVisionModel.from_pretrained(
    '/content/drive/MyDrive/numera-ml/models/qwen3vl-financial-lora',
    load_in_4bit=True,
)

EXPORT_DIR = '/content/drive/MyDrive/numera-ml/models/qwen3vl-financial-merged'
model.save_pretrained_merged(EXPORT_DIR, tokenizer, save_method='merged_16bit')
print(f'Merged model saved to {EXPORT_DIR}')


In [ ]:
# Cell 2: Quantize for deployment (4-bit GGUF)
# This creates a compact model for CPU/small GPU inference
GGUF_DIR = '/content/drive/MyDrive/numera-ml/models/qwen3vl-financial-gguf'
model.save_pretrained_gguf(
    GGUF_DIR, tokenizer,
    quantization_method='q4_k_m',
)
print(f'Quantized GGUF model saved to {GGUF_DIR}')


In [ ]:
# Cell 3: Register in MLflow
import mlflow

mlflow.set_tracking_uri('file:///content/drive/MyDrive/numera-ml/mlruns')

with mlflow.start_run(run_name='qwen3vl-export'):
    mlflow.log_artifact(EXPORT_DIR, 'model')
    result = mlflow.register_model(
        f'runs:/{mlflow.active_run().info.run_id}/model',
        'qwen3vl-financial'
    )

    from mlflow.tracking import MlflowClient
    client = MlflowClient()
    client.transition_model_version_stage(
        name='qwen3vl-financial',
        version=result.version,
        stage='Production',
    )

print(f'Model registered as qwen3vl-financial v{result.version} → Production')
print('Set OCR_VLM_MODEL_PATH to the exported model dir in your docker-compose.yml')
